# Validação dos dados de hemodiálise — SIA/SUS

**Objetivo:** verificar de forma reproduzível a estrutura, os períodos, as UFs, os valores e as totalizações do arquivo extraído do TabNet.

Este notebook valida o CSV em relação às regras documentadas no arquivo `mapeamento_hemodialise_sia_sus.md`. A tabela contém quantidade aprovada agregada por UF e mês; não identifica pessoas nem resultados clínicos.

> Execute as células na ordem. A validação usa apenas a biblioteca padrão do Python, sem necessidade de instalar pacotes.


## Fonte dos dados

Os dados foram obtidos no [TabNet DATASUS — Produção Ambulatorial do SUS por local de atendimento](https://tabnet.datasus.gov.br/cgi/deftohtm.exe?sia/cnv/qabr.def). Para reproduzir a extração, selecione Brasil, linha = UF, coluna = Ano/mês atendimento, conteúdo = Qtd.aprovada, período de janeiro/2024 a dezembro/2025 e os cinco procedimentos SIGTAP descritos no `mapeamento_hemodialise_sia_sus.md`.

O arquivo local validado neste notebook está em `Dados/hemodialise_sia_sus_uf_mes_2024_2025.csv`. A fonte oficial pode atualizar competências recentes; registre a data de cada nova extração.


## 1. Configuração e leitura

O arquivo está delimitado por ponto e vírgula (`;`) e usa nomes de meses em português. A primeira coluna contém código e nome da UF; a última linha contém o total Brasil.


In [1]:
from pathlib import Path
import csv
import re
import calendar

# Caminho relativo ao diretório do notebook.
CAMINHOS_CSV = [
    Path("../Dados/02_derivados_preliminares/SIA/hemodialise_sia_sus_uf_mes_2024_2025.csv"),
    Path("../Dados/hemodialise_sia_sus_uf_mes_2024_2025.csv"),
    Path("estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/Dados/hemodialise_sia_sus_uf_mes_2024_2025.csv"),
]
ARQUIVO_CSV = next((caminho for caminho in CAMINHOS_CSV if caminho.exists()), CAMINHOS_CSV[0])

# Lista oficial esperada das 27 UFs: código IBGE de dois dígitos e nome.
UFS_ESPERADAS = {
    "11": "Rondônia", "12": "Acre", "13": "Amazonas", "14": "Roraima",
    "15": "Pará", "16": "Amapá", "17": "Tocantins", "21": "Maranhão",
    "22": "Piauí", "23": "Ceará", "24": "Rio Grande do Norte",
    "25": "Paraíba", "26": "Pernambuco", "27": "Alagoas", "28": "Sergipe",
    "29": "Bahia", "31": "Minas Gerais", "32": "Espírito Santo",
    "33": "Rio de Janeiro", "35": "São Paulo", "41": "Paraná",
    "42": "Santa Catarina", "43": "Rio Grande do Sul", "50": "Mato Grosso do Sul",
    "51": "Mato Grosso", "52": "Goiás", "53": "Distrito Federal",
}

# Os meses do cabeçalho vêm abreviados em português. Essa tabela evita depender
# da configuração regional (locale) instalada no computador.
MESES_PT = {
    "Jan": 1, "Fev": 2, "Mar": 3, "Abr": 4, "Mai": 5, "Jun": 6,
    "Jul": 7, "Ago": 8, "Set": 9, "Out": 10, "Nov": 11, "Dez": 12,
}

if not ARQUIVO_CSV.exists():
    raise FileNotFoundError(
        f"Não encontrei {ARQUIVO_CSV}. Execute o notebook a partir da pasta Trabalho-final "
        "ou ajuste ARQUIVO_CSV para apontar ao arquivo."
    )

# utf-8-sig também remove eventual marca BOM no início do arquivo.
with ARQUIVO_CSV.open("r", encoding="utf-8-sig", newline="") as arquivo:
    linhas = list(csv.reader(arquivo, delimiter=";"))

print(f"Arquivo: {ARQUIVO_CSV}")
print(f"Linhas incluindo cabeçalho: {len(linhas)}")
print(f"Colunas no cabeçalho: {len(linhas[0]) if linhas else 0}")


Arquivo: ../Dados/02_derivados_preliminares/SIA/hemodialise_sia_sus_uf_mes_2024_2025.csv
Linhas incluindo cabeçalho: 29
Colunas no cabeçalho: 26


## 2. Validar cabeçalho e cobertura temporal

Verifica se existem exatamente os 24 meses de janeiro/2024 a dezembro/2025, em ordem, além da coluna de total. Também acusa delimitador errado, cabeçalhos repetidos ou linhas com quantidades diferentes de campos.


In [2]:
# Cria os rótulos mensais que deveriam constar, na ordem cronológica correta.
MESES_ESPERADOS = [
    f"{ano}/{mes_nome}"
    for ano in (2024, 2025)
    for mes_num, mes_nome in enumerate(MESES_PT, start=1)
]
CABECALHO_ESPERADO = ["UF", *MESES_ESPERADOS, "Total 2024-2025"]

assert linhas, "O arquivo está vazio."
cabecalho = linhas[0]
assert cabecalho == CABECALHO_ESPERADO, (
    "Cabeçalho diferente do esperado.\n"
    f"Esperado: {CABECALHO_ESPERADO}\n"
    f"Encontrado: {cabecalho}"
)
assert len(cabecalho) == len(set(cabecalho)), "Há nomes de colunas repetidos."

# Todas as linhas precisam ter a mesma quantidade de campos do cabeçalho.
larguras = {len(linha) for linha in linhas}
assert larguras == {len(cabecalho)}, (
    f"Linhas com número de colunas inconsistente: {sorted(larguras)}"
)

print("OK: cabeçalho, ordem dos 24 meses e largura das linhas conferidos.")


OK: cabeçalho, ordem dos 24 meses e largura das linhas conferidos.


## 3. Separar UFs e validar completude

O TabNet exporta rótulos como `11 Rondônia`; o código abaixo separa o código IBGE e o nome. A linha `Total` é removida do conjunto de UFs e tratada como controle de totalização.


In [3]:
# Separa a linha de total nacional das linhas estaduais.
linhas_dados = linhas[1:]
linhas_total = [linha for linha in linhas_dados if linha[0].strip().casefold() == "total"]
linhas_uf = [linha for linha in linhas_dados if linha[0].strip().casefold() != "total"]

assert len(linhas_total) == 1, (
    f"Esperada exatamente uma linha 'Total'; encontradas {len(linhas_total)}."
)
assert len(linhas_uf) == 27, f"Esperadas 27 UFs; encontradas {len(linhas_uf)}."

# Transforma cada linha em um registro estruturado.
registros = []
for numero_linha, linha in enumerate(linhas_uf, start=2):
    correspondencia = re.fullmatch(r"\s*(\d{2})\s+(.+?)\s*", linha[0])
    assert correspondencia, (
        f"Rótulo de UF inválido na linha {numero_linha}: {linha[0]!r}"
    )
    codigo, nome = correspondencia.groups()
    registros.append({
        "codigo_uf": codigo,
        "nome_uf": nome,
        "valores_mensais": linha[1:-1],
        "total_informado": linha[-1],
    })

codigos_encontrados = [registro["codigo_uf"] for registro in registros]
assert len(codigos_encontrados) == len(set(codigos_encontrados)), "Há códigos de UF duplicados."
assert set(codigos_encontrados) == set(UFS_ESPERADAS), (
    "Conjunto de códigos de UF divergente. "
    f"Faltantes: {sorted(set(UFS_ESPERADAS) - set(codigos_encontrados))}; "
    f"inesperados: {sorted(set(codigos_encontrados) - set(UFS_ESPERADAS))}."
)

# Acentos podem estar representados em formas Unicode diferentes; como checagem
# prática, comparamos nomes ignorando apenas caixa e espaços externos.
erros_nomes = [
    (r["codigo_uf"], UFS_ESPERADAS[r["codigo_uf"]], r["nome_uf"])
    for r in registros
    if r["nome_uf"].strip().casefold() != UFS_ESPERADAS[r["codigo_uf"]].casefold()
]
assert not erros_nomes, f"Nomes de UF divergentes: {erros_nomes}"

registro_total = linhas_total[0]
print("OK: 27 UFs únicas, códigos e nomes conferidos; linha total identificada.")


OK: 27 UFs únicas, códigos e nomes conferidos; linha total identificada.


## 4. Validar valores e recalcular totais

Cada competência e o total informado devem ser números inteiros não negativos. Em seguida, o notebook recalcula o total de cada UF e o total Brasil por mês, comparando os resultados ao CSV.


In [4]:
def converter_inteiro(texto, contexto):
    """Converte um campo para inteiro não negativo ou gera erro contextualizado."""
    valor = texto.strip()
    if valor == "":
        raise ValueError(f"Valor ausente em {contexto}.")
    try:
        numero = int(valor)
    except ValueError as erro:
        raise ValueError(f"Valor não inteiro em {contexto}: {texto!r}.") from erro
    if numero < 0:
        raise ValueError(f"Valor negativo em {contexto}: {numero}.")
    return numero

# Converte os valores mensais e o total por UF para inteiros.
for registro in registros:
    codigo = registro["codigo_uf"]
    registro["valores_mensais"] = [
        converter_inteiro(valor, f"UF {codigo}, {mes}")
        for mes, valor in zip(MESES_ESPERADOS, registro["valores_mensais"])
    ]
    registro["total_informado"] = converter_inteiro(
        registro["total_informado"], f"total informado da UF {codigo}"
    )

# Converte os valores da linha Total Brasil.
total_brasil_mensal = [
    converter_inteiro(valor, f"Total Brasil, {mes}")
    for mes, valor in zip(MESES_ESPERADOS, registro_total[1:-1])
]
total_brasil_informado = converter_inteiro(
    registro_total[-1], "Total Brasil 2024-2025"
)

# Cada total estadual deve corresponder à soma das 24 competências.
divergencias_totais_uf = []
for registro in registros:
    soma_meses = sum(registro["valores_mensais"])
    if soma_meses != registro["total_informado"]:
        divergencias_totais_uf.append({
            "UF": registro["nome_uf"],
            "soma_dos_meses": soma_meses,
            "total_informado": registro["total_informado"],
        })
assert not divergencias_totais_uf, (
    f"Totais anuais divergentes por UF: {divergencias_totais_uf}"
)

# Soma de todas as UFs em cada mês deve reproduzir a linha Total Brasil.
divergencias_mensais = []
for indice, mes in enumerate(MESES_ESPERADOS):
    soma_ufs = sum(r["valores_mensais"][indice] for r in registros)
    if soma_ufs != total_brasil_mensal[indice]:
        divergencias_mensais.append({
            "mes": mes,
            "soma_das_ufs": soma_ufs,
            "total_brasil_informado": total_brasil_mensal[indice],
        })
assert not divergencias_mensais, (
    f"Totais mensais do Brasil divergentes: {divergencias_mensais}"
)

# Compara o total geral calculado pelas UFs ao total geral informado pelo TabNet.
soma_geral_ufs = sum(r["total_informado"] for r in registros)
soma_geral_meses_brasil = sum(total_brasil_mensal)
assert soma_geral_ufs == total_brasil_informado, (
    f"Soma dos totais estaduais ({soma_geral_ufs}) difere do total Brasil "
    f"informado ({total_brasil_informado})."
)
assert soma_geral_meses_brasil == total_brasil_informado, (
    f"Soma dos meses da linha Brasil ({soma_geral_meses_brasil}) difere do "
    f"total Brasil informado ({total_brasil_informado})."
)

print("OK: valores inteiros não negativos; totais por UF, mês e Brasil conciliados.")
print(f"Total Brasil 2024-2025: {total_brasil_informado:,}".replace(",", "."))


OK: valores inteiros não negativos; totais por UF, mês e Brasil conciliados.
Total Brasil 2024-2025: 37.296.852


## 5. Conferir SIGTAP e filtros documentados

Os códigos SIGTAP não estão discriminados nas linhas do CSV: o arquivo é uma tabela agregada após aplicação dos filtros no TabNet. Portanto, esta célula confere a lista registrada no mapeamento do projeto; ela não substitui a conferência da tela/consulta original ou dos metadados da extração.


In [5]:
# Códigos cuja seleção está documentada em mapeamento_hemodialise_sia_sus.md.
CODIGOS_SIGTAP_ESPERADOS = {
    "0305010093", "0305010107", "0305010115",
    "0305010123", "0305010204",
}
CAMINHOS_MAPEAMENTO = [
    Path("../mapeamento_hemodialise_sia_sus.md"),
    Path("estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/mapeamento_hemodialise_sia_sus.md"),
]
ARQUIVO_MAPEAMENTO = next(
    (caminho for caminho in CAMINHOS_MAPEAMENTO if caminho.exists()),
    CAMINHOS_MAPEAMENTO[0],
)
assert ARQUIVO_MAPEAMENTO.exists(), f"Mapeamento não encontrado: {ARQUIVO_MAPEAMENTO}"
texto_mapeamento = ARQUIVO_MAPEAMENTO.read_text(encoding="utf-8")

# Extrai os códigos de dez dígitos descritos no arquivo de mapeamento.
codigos_documentados = set(re.findall(r"\b\d{10}\b", texto_mapeamento))
assert codigos_documentados == CODIGOS_SIGTAP_ESPERADOS, (
    "Lista de códigos no mapeamento difere da lista esperada. "
    f"Documentados: {sorted(codigos_documentados)}"
)

# Confere também os filtros/período descritos no registro da extração.
termos_filtro = ["Qtd.aprovada", "Ano/mês atendimento", "UF"]
termos_ausentes = [termo for termo in termos_filtro if termo not in texto_mapeamento]
assert not termos_ausentes, f"Filtros/medida sem registro no mapeamento: {termos_ausentes}"
for termo in ["Jan/2024", "Dez/2025"]:
    assert termo in texto_mapeamento, f"Período {termo} não registrado no mapeamento."

print(f"OK: {len(codigos_documentados)} códigos SIGTAP e filtros constam no mapeamento.")
print("Limite: a presença no mapeamento confirma o registro documental, não valida novamente a consulta no TabNet.")


OK: 5 códigos SIGTAP e filtros constam no mapeamento.
Limite: a presença no mapeamento confirma o registro documental, não valida novamente a consulta no TabNet.


## 6. Resumo final das verificações

Se as células anteriores terminaram sem `AssertionError`, as validações estruturais e aritméticas passaram. Esta célula reúne as métricas principais para facilitar o registro do controle de qualidade.


In [6]:
resumo_validacao = {
    "arquivo": str(ARQUIVO_CSV),
    "uf_unicas": len(registros),
    "meses": len(MESES_ESPERADOS),
    "colunas": len(cabecalho),
    "total_brasil_2024_2025": total_brasil_informado,
    "codigos_sigtap_documentados": len(codigos_documentados),
    "totais_uf_conciliados": len(divergencias_totais_uf) == 0,
    "totais_mensais_brasil_conciliados": len(divergencias_mensais) == 0,
}
for chave, valor in resumo_validacao.items():
    print(f"{chave}: {valor}")


arquivo: ../Dados/02_derivados_preliminares/SIA/hemodialise_sia_sus_uf_mes_2024_2025.csv
uf_unicas: 27
meses: 24
colunas: 26
total_brasil_2024_2025: 37296852
codigos_sigtap_documentados: 5
totais_uf_conciliados: True
totais_mensais_brasil_conciliados: True


## 7. Interpretação e limitações

- `Qtd.aprovada` representa produção administrativa aprovada no SIA/SUS; não é contagem de pacientes únicos, cobertura, adesão ou desfecho clínico.
- Os códigos SIGTAP e filtros são conferidos contra o registro de mapeamento. Como não estão individualizados no CSV, uma auditoria independente da seleção requer salvar/consultar os parâmetros originais do TabNet.
- A própria documentação do projeto alerta que os seis meses mais recentes podem sofrer atualização. Registre a data da extração e refaça esta validação após atualizar o arquivo.
- Uma validação aprovada atesta consistência interna e aderência às regras programadas; não demonstra, por si só, ausência de erros na fonte de origem.


## 7. Conferir os filtros no resultado salvo do TabNet

Como os códigos não aparecem em cada linha do CSV agregado, esta célula confere a página HTML salva da consulta, além do mapeamento escrito. Ela valida evidência registrada da consulta, não reexecuta a tabulação no servidor.


In [7]:
from pathlib import Path

CANDIDATOS_HTML = [
    Path("../Dados/01_fontes_originais/TabNet"),
    Path("../Dados"),
    Path("estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/Dados"),
]
PASTA_DADOS = next((p for p in CANDIDATOS_HTML if p.exists()), CANDIDATOS_HTML[0])
arquivos_html = list(PASTA_DADOS.glob("*.html"))
assert arquivos_html, f"Não encontrei o HTML salvo do TabNet em {PASTA_DADOS}."
texto_html = "\n".join(p.read_text(encoding="latin-1", errors="replace") for p in arquivos_html)

# A própria página salva da tabulação deve mostrar medida, eixo temporal,
# cobertura 2024–2025 e os cinco códigos selecionados.
for termo in ["Qtd.aprovada", "Ano/mês atendimento", "2024-2025"]:
    assert termo.casefold() in texto_html.casefold(), f"Filtro ausente no HTML: {termo}"
for codigo in CODIGOS_SIGTAP_ESPERADOS:
    assert codigo in texto_html, f"Código SIGTAP não localizado na consulta HTML: {codigo}"
print("OK: HTML salvo confirma período, medida e os cinco códigos SIGTAP.")
print("HTML(s) conferido(s):", [p.name for p in arquivos_html])


OK: HTML salvo confirma período, medida e os cinco códigos SIGTAP.
HTML(s) conferido(s): ['TabNet_producao_ambulatorial.html']


## 8. Conclusão da validação

Uma validação aprovada indica consistência estrutural e aritmética do arquivo organizado, e compatibilidade com os filtros visíveis na evidência HTML e no mapeamento. Não substitui auditoria dos registros fonte do DATASUS nem implica interpretação clínica.
